In [ ]:
# Enable autoreload to automatically reload modules before executing code
%load_ext autoreload
%autoreload 2

# Introduction to Time Series Forecasting with Machine Learning

Time series forecasting is the process of predicting future values based on previously observed data points, where each observation is recorded at a specific time. This is a fundamental task in many fields, such as energy demand prediction, stock price forecasting, weather modeling, and more.

## What Makes Time Series Special?

Unlike typical datasets, time series data is ordered in time, and each value can depend on previous values. This temporal dependency means that traditional machine learning methods must be adapted to handle the unique structure of time series.

## Our Approach: Forecasting with Lagged Features

To start, we will focus on **univariate forecasting**—predicting future values of a single variable (the "target"), using only its own past values. We will not include any external (exogenous) variables yet. Instead, we will create **lag features**, which are simply previous values of the target variable. For example, to predict the value at time $t$, we might use the values at times $t-1$, $t-2$, $t-48$, etc., as input features.

This approach allows us to use powerful machine learning models that are typically designed for tabular data, by transforming our time series into a supervised learning problem.

## Models We Will Explore

We will experiment with a range of machine learning models, from simple linear models to advanced tree-based algorithms:

- **Linear Regression**: A straightforward model that predicts the target as a weighted sum of lagged values.
- **Lasso Regression**: A linear model with L1 regularization, which can automatically select the most important lags by shrinking less useful coefficients to zero.
- **Ridge Regression**: Similar to linear regression, but with L2 regularization to prevent overfitting by penalizing large coefficients.
- **Decision Tree Models**:
    - **LightGBM**
    - **XGBoost**
    - **CatBoost**

  These are powerful gradient boosting algorithms that can capture complex, non-linear relationships in the data.

## Why Start Without Exogenous Variables?

By focusing only on the target variable and its lags, we can:

- Build intuition for how past values influence future predictions.
- Understand the strengths and limitations of different machine learning models in a controlled setting.
- Lay the groundwork for more advanced models that incorporate additional features later.

---

In the following sections, we will walk through the process of preparing lagged features, training each model, and evaluating their forecasting performance. Each step will be explained in detail, with code examples and visualizations to help you build a strong foundation in time series forecasting with machine learning.

In [ ]:
import polars as pl
import plotly.express as px
import seaborn as sns
from utilsforecast.plotting import plot_series
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate
from mlforecast import MLForecast
from mlforecast.target_transforms import (
    LocalStandardScaler,
    LocalMinMaxScaler,
    Differences,
)
from plotting_utils import (
    plotly_series as plot_series,
    plot_acf,
    plot_residuals_diagnostic,
)
from summary_utils import get_fitted_residuals
from statsmodels.stats.diagnostic import acorr_ljungbox
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

import plotly.graph_objects as go


In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select(
        [
            time_,
            id_,
            target_,
            # "Acorn",
            # "Acorn_grouped",
            # "holidays",
            # "visibility",
            # "windBearing",
            # "temperature",
            # "dewPoint",
            # "pressure",
            # "apparentTemperature",
            # "windSpeed",
            # "precipType",
            # "icon",
            # "humidity",
            # "summary",
        ]
    )
    .explode(
        [
            time_,
            target_,
            # "holidays",
            # "visibility",
            # "windBearing",
            # "temperature",
            # "dewPoint",
            # "pressure",
            # "apparentTemperature",
            # "windSpeed",
            # "precipType",
            # "icon",
            # "humidity",
            # "summary",
        ]
    )
)
data.head()

In [ ]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

In [ ]:
from mlforecast.lag_transforms import (
    RollingMean,
    RollingStd,
    SeasonalRollingMean,
    SeasonalRollingStd,
    ExponentiallyWeightedMean,
)

lags = [1, 2, 48, 336]
lag_transforms = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        RollingMean(window_size=48),
        RollingStd(window_size=3),
        RollingStd(window_size=6),
        RollingStd(window_size=12),
        RollingStd(window_size=48),
        ExponentiallyWeightedMean(alpha=0.25),
    ],
    48: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        SeasonalRollingMean(season_length=48, window_size=3),
        SeasonalRollingStd(season_length=48, window_size=3),
    ],
    336: [
        RollingMean(window_size=4),
        RollingMean(window_size=8),
        RollingStd(window_size=4),
        RollingStd(window_size=8),
        SeasonalRollingMean(season_length=336, window_size=3),
        SeasonalRollingStd(season_length=336, window_size=3),
    ],
}

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

mlf.preprocess(data).head(2)

# Introduction to Linear Regression for Time Series Forecasting

Linear regression is one of the most fundamental and widely used techniques in statistics and machine learning. It helps us understand and model the relationship between a **dependent variable** (what we want to predict) and one or more **independent variables** (the features we use for prediction).

In the context of **time series forecasting**, we often use linear regression to predict future values of a series based on its own past values—these are called **lagged features**.

---

## What is Linear Regression?

At its core, linear regression fits a straight line to the data, modeling the relationship as:

$$
\hat{y}_t = \beta_0 + \beta_1 x_{t,1} + \beta_2 x_{t,2} + \cdots + \beta_p x_{t,p}
$$

- $\hat{y}_t$ is the predicted value at time $t$.
- $\beta_0$ is the intercept (the value when all features are zero).
- $\beta_1, \beta_2, ..., \beta_p$ are the coefficients (weights) for each feature.
- $x_{t,1}, x_{t,2}, ..., x_{t,p}$ are the feature values at time $t$ (for time series, these are often lagged values like $y_{t-1}, y_{t-2}$, etc.).

**Analogy:**  
Think of linear regression as drawing the best possible straight line through a cloud of points, so that the line is as close as possible to all the points.

---

## Why Use Linear Regression for Time Series?

- **Simplicity:** Easy to understand and interpret.
- **Speed:** Fast to train, even on large datasets.
- **Baseline:** Provides a strong baseline to compare more complex models against.
- **Interpretability:** The coefficients tell us how much each lagged value influences the forecast.

---

## How Does Linear Regression Work with Time Series?

For time series, we transform the data so that each row contains the value we want to predict ($y_t$) and its lagged values ($y_{t-1}, y_{t-2}, ...$). The model then learns how past values relate to the current value.

**Example:**  
Suppose we want to predict energy consumption at time $t$ using the previous two time steps:

| $y_{t-2}$ | $y_{t-1}$ | $y_t$ (target) |
|-----------|-----------|---------------|
|   0.368   |   0.386   |     0.170     |
|   0.386   |   0.170   |     0.021     |
|   0.170   |   0.021   |     0.038     |

Here, $y_{t-2}$ and $y_{t-1}$ are the features, and $y_t$ is what we want to predict.

---

## Common Beginner Questions

**Q: Can linear regression capture seasonality or trends?**  
A: By itself, linear regression is limited to modeling linear relationships. However, by including lagged features that reflect seasonality (e.g., $y_{t-48}$ for daily seasonality in half-hourly data), we can help the model capture these patterns.

**Q: What if the relationship is not linear?**  
A: Linear regression may not perform well if the true relationship is non-linear. In such cases, more advanced models (like tree-based methods) may be better, but linear regression is still a valuable starting point.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[LinearRegression()],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

y_hat = mlf.cross_validation(
    data,
    h=48,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals.get_column(time_)
residuals = fitted_residuals.get_column("LinearRegression")

plot_residuals_diagnostic(residuals, time)

In [ ]:
resid_test = acorr_ljungbox(residuals)
resid_test

In [ ]:
mlf.fit(data)

In [ ]:
fitted_model = mlf.models_["LinearRegression"]

In [ ]:
dir(fitted_model)

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.coef_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Coefficients from Linear Regression",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Linear Regression Feature Coefficients",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(data).to_pandas().drop(columns=[id_, time_, target_])

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

# Understanding Regularized Linear Regression for Time Series Forecasting

Now that we've explored standard linear regression, let's take a step further and discuss **regularized linear regression**. Regularization is a crucial concept in machine learning, especially when dealing with many features (like our lagged and rolling statistics), as it helps prevent **overfitting**.

---

## What is Regularization?

**Regularization** adds a penalty to the model's loss function to discourage it from fitting the noise in the data. This is especially important in time series, where many lagged features can lead to a model that fits the training data very well but performs poorly on new, unseen data.

### Why Do We Need Regularization?

- **Overfitting** occurs when a model learns not only the underlying pattern but also the random fluctuations (noise) in the training data.
- With many features, especially if some are not very informative, the model can assign large weights to irrelevant features, making predictions unstable.
- Regularization helps by "shrinking" the coefficients of less important features towards zero, making the model simpler and more robust.

---

## Types of Regularized Linear Regression

There are two main types of regularized linear regression:

### 1. **Ridge Regression (L2 Regularization)**

- **How it works:** Adds a penalty proportional to the **sum of the squares** of the coefficients.
- **Mathematical formula:**

    $$
    \text{Loss} = \sum_{t} (y_t - \hat{y}_t)^2 + \lambda \sum_{j} \beta_j^2
    $$

    - $\lambda$ is the regularization strength (higher $\lambda$ means more penalty).
    - The penalty discourages large coefficients but does **not** force them to zero.

- **Analogy:** Imagine stretching a rubber band over the data points. Ridge regression keeps the band taut, preventing it from bending too much to fit every point.

### 2. **Lasso Regression (L1 Regularization)**

- **How it works:** Adds a penalty proportional to the **sum of the absolute values** of the coefficients.
- **Mathematical formula:**

    $$
    \text{Loss} = \sum_{t} (y_t - \hat{y}_t)^2 + \lambda \sum_{j} |\beta_j|
    $$

    - Lasso can shrink some coefficients **exactly to zero**, effectively performing feature selection.

- **Analogy:** Lasso acts like a tight leash, pulling some coefficients all the way down to zero, so only the most important features remain.

---

## When Should You Use Regularization?

- When you have **many features** (like lots of lags or rolling statistics).
- When you notice your model performs much better on training data than on validation/test data.
- When you want a **simpler, more interpretable model** (especially with Lasso).

---

## Common Beginner Questions

**Q: How do I choose between Ridge and Lasso?**  
A: If you suspect many features are irrelevant, Lasso is helpful because it can set their coefficients to zero. If you believe all features are useful but want to prevent any from dominating, Ridge is a good choice.

**Q: How do I pick the regularization strength ($\lambda$)?**  
A: Typically, you use **cross-validation** to find the value of $\lambda$ that gives the best performance on unseen data.

---

In the next section, we'll see how to implement Ridge and Lasso regression for our time series forecasting problem using the `mlforecast` library, and compare their results to standard linear regression.

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        LassoCV(),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

y_hat = mlf.cross_validation(
    data,
    h=48,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals.get_column(time_)
residuals = fitted_residuals.get_column("Ridge")

plot_residuals_diagnostic(residuals, time)

In [ ]:
resid_test = acorr_ljungbox(residuals)
resid_test

In [ ]:
mlf.fit(data)

In [ ]:
fitted_model = mlf.models_["Ridge"]

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.coef_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Coefficients from Linear Regression",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Linear Regression Feature Coefficients",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(data).to_pandas().drop(columns=[id_, time_, target_])

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        LassoCV(),
    ],
    freq="30m",
    target_transforms=[LocalStandardScaler()],
    lags=lags,
    lag_transforms=lag_transforms,
)

y_hat = mlf.cross_validation(
    data,
    h=48,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals.get_column(time_)
residuals = fitted_residuals.get_column("Ridge")

plot_residuals_diagnostic(residuals, time)

# Introduction to Decision Trees for Time Series Forecasting

Decision trees are a powerful and intuitive type of machine learning model that can capture complex, non-linear relationships in data. They are especially useful when the relationship between features and the target variable is not simply linear, as is often the case in real-world time series.

---

## What is a Decision Tree?

A **decision tree** is a model that makes predictions by learning a series of "if-then" rules from the data. Imagine a flowchart where each internal node asks a question about one of the features (e.g., "Is lag1 > 0.5?"), and each branch leads to another question or a final prediction.

- **Root node:** The starting point, where the first question is asked.
- **Internal nodes:** Each node splits the data based on a feature and a threshold.
- **Leaf nodes:** The endpoints, where a prediction is made (e.g., the average value of the target in that group).

**Analogy:**  
Think of a decision tree as a game of 20 Questions, where each question narrows down the possibilities until you reach an answer.

---

## Why Use Decision Trees for Time Series?

- **Non-linear relationships:** Trees can model interactions and non-linear effects between lagged features and the target.
- **Automatic feature selection:** Trees naturally select the most important features for splitting.
- **Interpretability:** You can visualize the tree to understand how decisions are made.
- **No need for feature scaling:** Unlike linear models, trees do not require normalization or standardization of features.

---

## How Do Decision Trees Work with Time Series Data?

Just like with linear models, we use lagged features (e.g., $y_{t-1}$, $y_{t-48}$, rolling means, etc.) as inputs. The tree learns which lags and statistics are most useful for predicting the next value.

**Example:**  
Suppose the tree learns that if the value 48 time steps ago ($y_{t-48}$) is high, and the recent rolling mean is low, then the next value is likely to be low. These kinds of rules can capture seasonal effects and sudden changes.

---

## Limitations and Extensions

- **Overfitting:** Single decision trees can easily overfit, especially with many features. This means they might perform well on training data but poorly on new data.
- **Solution:** In practice, we often use **ensemble methods** like Random Forests or Gradient Boosted Trees (e.g., LightGBM, XGBoost, CatBoost) to combine many trees and improve performance and robustness.

---

## Common Beginner Questions

**Q: Can decision trees handle missing values?**  
A: Many tree implementations can handle missing values natively, but it's still good practice to fill or impute missing lags.

**Q: Are trees sensitive to outliers?**  
A: Trees are generally robust to outliers, since splits are based on feature thresholds rather than averages.

**Q: How do trees compare to linear models?**  
A: Trees can capture more complex patterns, but may require more data and careful tuning to avoid overfitting.

---

In the next section, we'll see how to apply decision tree-based models to our time series forecasting problem, starting with simple trees and then moving to advanced ensemble methods.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
mlf = MLForecast(
    models=[
        DecisionTreeRegressor(max_depth=4, random_state=42),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

y_hat = mlf.cross_validation(
    data,
    h=48,
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals.get_column(time_)
residuals = fitted_residuals.get_column("DecisionTreeRegressor")

plot_residuals_diagnostic(residuals, time)

In [ ]:
resid_test = acorr_ljungbox(residuals)
resid_test

In [ ]:
mlf.fit(data)

In [ ]:
fitted_model = mlf.models_["DecisionTreeRegressor"]

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.feature_importances_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Importances from Decision Tree Regressor",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Feature Importances from Decision Tree Regressor",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(data).to_pandas().drop(columns=[id_, time_, target_])

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

# Introduction to Random Forests and Bagging for Time Series Forecasting

Ensemble methods are a powerful class of machine learning techniques that combine the predictions of multiple models to achieve better performance than any single model alone. Two of the most important ensemble concepts are **bagging** and **random forests**.

---

## What is Bagging?

**Bagging** stands for *Bootstrap Aggregating*. It is a technique designed to improve the stability and accuracy of machine learning algorithms, especially those that are prone to overfitting (like decision trees).

### How Bagging Works

1. **Bootstrap Sampling:**  
    - Create multiple new training datasets by randomly sampling (with replacement) from the original data. Each sample is called a *bootstrap sample*.
2. **Model Training:**  
    - Train a separate model (often a decision tree) on each bootstrap sample.
3. **Aggregation:**  
    - For regression, average the predictions from all models.
    - For classification, use majority voting.

**Analogy:**  
Imagine asking several people (each with slightly different experiences) to predict tomorrow’s weather. By averaging their answers, you often get a more reliable forecast than relying on just one person.

### Why Does Bagging Help?

- **Reduces Variance:** By averaging many models, bagging smooths out the "noise" from individual models, leading to more robust predictions.
- **Prevents Overfitting:** Especially useful for models like decision trees, which can fit the training data too closely.

---

## What is a Random Forest?

A **Random Forest** is an extension of bagging that adds an extra layer of randomness to further improve predictive power.

### How Random Forests Work

- **Multiple Decision Trees:** Like bagging, random forests train many decision trees on different bootstrap samples.
- **Random Feature Selection:**  
  - At each split in a tree, only a random subset of features is considered for splitting, rather than all features.
  - This decorrelates the trees, making the ensemble even more robust.

**Mathematical Intuition:**  
If each tree predicts $\hat{y}_t^{(i)}$, the random forest prediction is:
$$
\hat{y}_t = \frac{1}{N} \sum_{i=1}^N \hat{y}_t^{(i)}
$$
where $N$ is the number of trees.

### Key Benefits

- **Handles Non-Linearity:** Captures complex, non-linear relationships in the data.
- **Feature Importance:** Provides estimates of which features (lags, rolling means, etc.) are most useful for prediction.
- **Robustness:** Less sensitive to outliers and noise compared to single trees.

---

## Why Use Random Forests for Time Series?

- **Captures Complex Patterns:** Can model interactions and non-linear effects between lagged features.
- **Reduces Overfitting:** Ensemble approach makes predictions more stable.
- **No Need for Feature Scaling:** Works well with raw or engineered features.

---

## Common Beginner Questions

**Q: Can random forests extrapolate future trends?**  
A: Random forests are great at capturing patterns seen in the training data, but they may struggle to extrapolate beyond the range of observed values.

**Q: How do I choose the number of trees?**  
A: More trees generally improve performance up to a point, but with diminishing returns. Common values are 100–500 trees.

**Q: Are random forests slow?**  
A: Training can be slower than a single tree, but predictions are usually fast, and training can be parallelized.

---

In the next section, we'll see how to apply random forests to our time series forecasting problem using the `mlforecast` library, and how to interpret their results.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# --- Random Forests for Time Series Forecasting ---

# Random Forest is an ensemble method that builds many decision trees and averages their predictions.
# This helps reduce overfitting and captures more complex patterns than a single tree.

# Let's set up MLForecast with a RandomForestRegressor.
mlf = MLForecast(
    models=[RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,  # Move the window by 1 step each time
    n_windows=1,  # Use a single validation window for simplicity
    fitted=True,  # Store fitted values for diagnostics
).drop("cutoff")

# Explanation:
# - RandomForestRegressor: Builds an ensemble of decision trees for regression.
# - n_estimators=100: Uses 100 trees for robust averaging.
# - max_depth=6: Limits tree depth to prevent overfitting.
# - random_state=42: Ensures reproducibility.
# - cross_validation: Provides a realistic estimate of forecasting accuracy.

y_hat.head()

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals.get_column(time_)
residuals = fitted_residuals.get_column("RandomForestRegressor")

plot_residuals_diagnostic(residuals, time)

In [ ]:
resid_test = acorr_ljungbox(residuals)
resid_test

In [ ]:
mlf.fit(data)

In [ ]:
fitted_model = mlf.models_["RandomForestRegressor"]

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.feature_importances_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Importances from Decision Tree Regressor",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Feature Importances from Decision Tree Regressor",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(data).to_pandas().drop(columns=[id_, time_, target_])

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

# Introduction to Gradient Boosting Trees, Boosting, and XGBoost/LightGBM

Gradient boosting trees are among the most powerful and popular machine learning algorithms for both tabular and time series data. They are widely used in industry and data science competitions due to their high predictive accuracy and flexibility.

---

## What is Boosting?

**Boosting** is an ensemble technique that combines the predictions of several "weak" models (models that perform just slightly better than random guessing) to create a strong overall model.

### How Does Boosting Work?

- **Sequential Learning:** Models are trained one after another. Each new model tries to correct the mistakes made by the previous models.
- **Weighted Focus:** More attention is given to data points that previous models predicted poorly.
- **Final Prediction:** The predictions from all models are combined (usually by weighted sum) to produce the final output.

**Analogy:**  
Imagine a group of students taking turns to answer quiz questions. Each student learns from the mistakes of the previous one, so the group as a whole gets better and better.

---

## What are Gradient Boosting Trees?

**Gradient Boosting Trees** use decision trees as the "weak" models in the boosting process. Each tree is trained to predict the errors (residuals) of the combined previous trees.

### Mathematical Intuition

At each step $m$, a new tree $h_m(x)$ is fit to the negative gradient (the direction of steepest descent) of the loss function:

$$
F_{m}(x) = F_{m-1}(x) + \gamma_m h_m(x)
$$

- $F_{m}(x)$: The boosted model after $m$ steps.
- $h_m(x)$: The new decision tree trained on the residuals.
- $\gamma_m$: Step size (learning rate).

This process continues for a set number of trees or until the model stops improving.

---

## Why Use Gradient Boosting Trees for Time Series?

- **Handles Non-Linearity:** Captures complex, non-linear relationships between lagged features and the target.
- **Automatic Feature Selection:** Learns which lags and statistics are most important.
- **Robust to Outliers:** Less sensitive to extreme values than linear models.
- **High Accuracy:** Often outperforms other algorithms on structured data.

---

## Popular Implementations: XGBoost and LightGBM

### XGBoost

- **Stands for:** eXtreme Gradient Boosting.
- **Key Features:** Highly optimized, supports regularization, handles missing values, parallel computation.
- **Widely Used:** Frequently wins machine learning competitions.

### LightGBM

- **Stands for:** Light Gradient Boosting Machine.
- **Key Features:** Faster training, lower memory usage, supports categorical features natively, efficient for large datasets.
- **Innovations:** Uses histogram-based algorithms and leaf-wise tree growth for speed and accuracy.

---

## Common Beginner Questions

**Q: How is boosting different from bagging (like Random Forests)?**  
A: Bagging builds models independently and averages their predictions, reducing variance. Boosting builds models sequentially, each focusing on correcting the previous, reducing bias.

**Q: What is the "learning rate"?**  
A: The learning rate ($\gamma$) controls how much each new tree contributes. Lower values mean slower, more careful learning (often leading to better results).

**Q: Can boosting overfit?**  
A: Yes, if too many trees are added or trees are too deep. Regularization and early stopping help prevent this.

---

In the next section, we'll see how to apply XGBoost and LightGBM to our time series forecasting problem using the `mlforecast` library, and how to interpret their results.

In [ ]:
from xgboost import XGBRegressor

# --- Gradient Boosting with XGBoost for Time Series Forecasting ---

# XGBoost is a powerful gradient boosting algorithm that builds trees sequentially,
# each one correcting the errors of the previous. It's widely used for its speed and accuracy.

# Let's set up MLForecast with an XGBRegressor.
mlf = MLForecast(
    models=[
        XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method="hist",
        )
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

# Explanation:
# - XGBRegressor: Implements gradient boosting with decision trees.
# - n_estimators=100: Number of boosting rounds (trees).
# - max_depth=6: Maximum depth of each tree (controls complexity).
# - learning_rate=0.1: Step size shrinkage to prevent overfitting.
# - tree_method="hist": Fast histogram-based algorithm for efficiency.
# - random_state=42: Ensures reproducibility.

y_hat.head()

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals.get_column(time_)
residuals = fitted_residuals.get_column("XGBRegressor")

plot_residuals_diagnostic(residuals, time)

In [ ]:
resid_test = acorr_ljungbox(residuals)
resid_test

In [ ]:
mlf.fit(data)

In [ ]:
fitted_model = mlf.models_["XGBRegressor"]

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.feature_importances_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Importances from Decision Tree Regressor",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Feature Importances from Decision Tree Regressor",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(data).to_pandas().drop(columns=[id_, time_, target_])

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

In [ ]:
from lightgbm import LGBMRegressor
from mlforecast.lgb_cv import LightGBMCV
# --- Gradient Boosting with LightGBM for Time Series Forecasting ---

# LightGBM is another state-of-the-art gradient boosting library, known for its speed and efficiency,
# especially with large datasets and many features. It uses decision trees as base learners,
# just like XGBoost, but with some algorithmic innovations for faster training.

# Let's set up MLForecast with an LGBMRegressor.
mlf = MLForecast(
    models=[
        LGBMRegressor(
            n_estimators=100,  # Number of boosting rounds (trees)
            max_depth=6,  # Maximum depth of each tree
            learning_rate=0.1,  # Step size shrinkage
            random_state=42,  # For reproducibility
        )
    ],
    freq="30min",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data.to_pandas(),
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,  # Move the window by 1 step each time
    n_windows=1,  # Use a single validation window for simplicity
    fitted=True,  # Store fitted values for diagnostics
).drop(columns="cutoff")

# Explanation:
# - LGBMRegressor: Implements gradient boosting with decision trees (LightGBM).
# - n_estimators=100: Number of boosting rounds (trees).
# - max_depth=6: Maximum depth of each tree (controls complexity).
# - learning_rate=0.1: Step size shrinkage to prevent overfitting.
# - random_state=42: Ensures reproducibility.

y_hat = pl.from_pandas(y_hat)

## Decoding the LightGBM Warning: `[LightGBM] [Warning] No further splits with positive gain, best gain: -inf`

This is a common warning you'll see when training LightGBM models. It generally means the tree-building process can't find more beneficial ways to split the data.

### How Tree-Based Models Learn

LightGBM builds an ensemble of decision trees. Each tree is constructed by finding the "best" way to split the data. The "best" split is determined by a metric called **gain**.

* **Splitting:** At each tree node, the algorithm looks at features and potential split points.
* **Gain:** It calculates how much the "purity" of data in child nodes would improve if a split were made. Higher gain means a better split.
* **Positive Gain:** Means a split would improve model performance (reduce error, increase accuracy).

### Interpreting the Warning

This warning means:

1.  **`No further splits with positive gain`**: The algorithm tried to find more ways to split, but no potential splits resulted in a positive gain. Any further splits would make the model worse or offer no improvement.
2.  **`best gain: -inf`**: This confirms the first point. `-inf` means the best gain found was a loss or no improvement.

### Why This Warning Occurs

This warning can occur for several reasons, and it's not always bad:

1.  **Tree Depth/Complexity Limits:**
    * **`max_depth`**: If set small, trees stop growing, and no further positive gain splits are found at leaves.
    * **`min_child_samples` / `min_data_in_leaf`**: If a split results in child nodes with too few samples, it's disallowed. This often leads to no positive gain splits as nodes shrink.
    * **`min_gain_to_split`**: If this threshold is set high, no splits will occur if they don't meet it.

2.  **Data Characteristics:**
    * **Homogeneous Nodes:** If data in a node is already very pure, there's little room for improvement.
    * **Too Few Samples:** As trees grow, nodes have fewer samples, making it hard to find statistically significant splits.
    * **Sparse Features:** Features with few unique values or low variance in a node might not allow good splits.

3.  **Regularization:**
    * LightGBM's default regularization (`lambda_l1`, `lambda_l2`) can make it harder to find splits with positive gain, especially for marginal gains.

### Is This Warning a Problem?

**Not necessarily.**

* **Often Harmless:** It often means LightGBM has reached a stable point for a given tree based on your hyperparameters. It prevents overfitting by avoiding unnecessary splits.
* **Can Indicate Underfitting (Less Common):** If it appears *very early* in training, or if overall model performance is poor, it *might* suggest underfitting. This means the model can't find good splits, pointing to a lack of predictive power from features or insufficient model complexity.

### What to Do (or Not Do) About It

1.  **Check Model Performance:** Evaluate your model on a validation set. If it's performing well, the warning is likely harmless.
2.  **Review Hyperparameters:**
    * **`max_depth`**: Consider increasing if very small.
    * **`min_child_samples` / `min_data_in_leaf`**: Consider decreasing if very large (but be careful of overfitting).
    * **`min_gain_to_split`**: If explicitly set high, try reducing it.
3.  **Feature Engineering:** If your model underperforms *and* you see this warning frequently, your features might not be rich enough for effective splits. Consider adding more `lag_transforms` or relevant external features for electricity demand.
4.  **Don't Over-Optimize:** Don't try to eliminate the warning if your model performs well. Forcing larger trees can lead to overfitting.

In your electricity demand forecasting, this warning is typically fine if your evaluation metrics (like RMSE or MAE) are good. It means LightGBM is being cautious and not making detrimental splits, which helps prevent overfitting. However, if performance is unsatisfactory, this warning, combined with your Ljung-Box test results, suggests a need for more informative features.

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
insample_forecasts = mlf.cross_validation_fitted_values()

In [ ]:
fitted_residuals = get_fitted_residuals(mlf)
time = fitted_residuals["ds"]
residuals = fitted_residuals["LGBMRegressor"]

plot_residuals_diagnostic(residuals, time)

In [ ]:
resid_test = acorr_ljungbox(residuals)
resid_test

In [ ]:
mlf.fit(data.to_pandas())

In [ ]:
fitted_model = mlf.models_["LGBMRegressor"]

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.feature_importances_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Importances from Decision Tree Regressor",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Feature Importances from Decision Tree Regressor",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(data.to_pandas()).drop(columns=[id_, time_, target_])

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500], check_additivity=False)

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

## Comparing Model Performance: Why XGBoost and Regularized Regression Stand Out

After evaluating several machine learning models for time series forecasting—including standard linear regression, decision trees, random forests, and gradient boosting methods—it's clear that **XGBoost** and **regularized regression models** (Ridge and Lasso) consistently deliver superior forecasting accuracy on our electricity demand data.

### Why Do XGBoost and Regularized Regression Perform Better?

- **XGBoost** excels because it can capture complex, non-linear relationships and interactions between lagged features, rolling statistics, and seasonal patterns. Its boosting approach sequentially corrects errors, leading to highly accurate predictions.
- **Regularized regression** (Ridge and Lasso) outperforms standard linear regression by penalizing overly complex models. This helps prevent overfitting, especially when we have many lagged and rolling features, and can even perform automatic feature selection (Lasso).

### Evidence from Our Results

- Both XGBoost and regularized regression models show lower error metrics (such as MAE, RMSE, and MASE) compared to simpler models.
- Residual diagnostics and statistical tests (like the Ljung-Box test) indicate that these models leave less autocorrelation in the residuals, meaning they capture more of the underlying structure in the data.

### Next Steps: Enhancing Our Models

Given these findings, we will **focus our modeling efforts on XGBoost and regularized regression**. Our next step is to **add more features** to the model, such as:

- **Calendar effects** (e.g., hour of day, day of week, holidays)
- **Weather variables** (e.g., temperature, humidity)
- **Additional engineered features** that may help capture patterns not yet modeled

By enriching our feature set, we aim to further improve forecasting accuracy and gain deeper insights into the drivers of electricity demand.

---

*Stay tuned as we expand our models and explore the impact of new features on time series forecasting performance!*

In [ ]:
date_features = [
    "month",
    "quarter",
    "week",
    "day",
    "weekday",
    "hour",
    "minute",
]

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method="hist",
        ),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=date_features,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

In [ ]:
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial

In [ ]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data,
    features=features,
    freq="30m",
    h=48,  # Horizon for future features
)
data_fourier.head(10)

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method="hist",
        ),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data_fourier,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,
    n_windows=1,
    fitted=True,
    static_features=[],
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)